# Day 31: LangGraph Tool Agent

Day 30 gave us nodes and edges. But a real agent needs **tools** — and the ability to decide when to use them.

Today we build a ReAct agent as an explicit graph. Every decision point is visible.

## Install

In [ ]:
%pip install langgraph langchain-google-genai langchain --quiet

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"]

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
print("✅ Model ready")

## Define Tools

In [ ]:
from langchain.tools import tool
import requests

HEADERS = {"User-Agent": "genai-50-days-demo/1.0"}

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    resp = requests.get(f"https://wttr.in/{city}?format=j1", headers=HEADERS)
    data = resp.json()["current_condition"][0]
    return f"{city}: {data['temp_C']}°C, {data['weatherDesc'][0]['value']}"

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression safely. Example: '2 + 3 * 4'"""
    allowed = set('0123456789+-*/.() ')
    if not all(c in allowed for c in expression):
        return "Error: only numbers and +-*/ allowed"
    return f"{expression} = {eval(expression)}"

tools = [get_weather, calculate]
print(f"✅ {len(tools)} tools defined: {[t.name for t in tools]}")

## Bind Tools to Model

In [ ]:
model_with_tools = model.bind_tools(tools)
print("✅ Model now knows about both tools")

## Define State

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

print("✅ State schema defined")

## Define Nodes

In [ ]:
from langchain_core.messages import ToolMessage

def llm_node(state: State):
    """Call the LLM — it decides whether to use a tool."""
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tool_node(state: State):
    """Execute whatever tool the LLM requested."""
    results = []
    for call in state["messages"][-1].tool_calls:
        tool_fn = {t.name: t for t in tools}[call["name"]]
        output = tool_fn.invoke(call["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
    return {"messages": results}

print("✅ Two nodes: llm_node, tool_node")

## Routing Function

In [ ]:
from typing import Literal
from langgraph.graph import END

def should_continue(state: State) -> Literal["tool_node", "__end__"]:
    """If the LLM made tool calls, route to tool_node. Otherwise, end."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tool_node"
    return END

print("✅ Router: tool calls → tool_node, no calls → end")

## Build the Graph

In [ ]:
from langgraph.graph import StateGraph, START

builder = StateGraph(State)

builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)

builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", should_continue, ["tool_node", END])
builder.add_edge("tool_node", "llm_node")

agent = builder.compile()
print("✅ Agent graph compiled")

## Visualize

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

## Test: Weather Query

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Berlin?"}]})

for msg in result["messages"]:
    msg.pretty_print()

## Test: Math Query

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is 15 * 37 + 42?"}]})

for msg in result["messages"]:
    msg.pretty_print()

## Test: No Tool Needed

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is the capital of Japan?"}]})

for msg in result["messages"]:
    msg.pretty_print()

## Add Memory

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent_with_memory = builder.compile(checkpointer=InMemorySaver())
print("✅ Same graph, now with memory")

In [ ]:
config = {"configurable": {"thread_id": "demo-1"}}

result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Tokyo?"}]},
    config
)
print(result["messages"][-1].content)

In [ ]:
result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "How about in Celsius, is that warm?"}]},
    config
)
print(result["messages"][-1].content)

## Key Takeaways

1. **`bind_tools`** teaches the model about tools — **`add_conditional_edges`** lets the graph decide when to use them
2. The **`should_continue`** router is the core decision: tool calls loop back, no calls end the graph
3. Adding memory is one line — **`compile(checkpointer=InMemorySaver())`** — same graph, persistent conversations